In [4]:
import pandas as pd
import numpy as np
from datetime import timedelta
import time # Assuming a training loop needs time measurement

# =====================================================
# LOAD DATA
# =====================================================
try:
    periodic_df = pd.read_excel("PERIODIC16-6-2024 LIMIT 5000.xlsx")
    event_df = pd.read_excel("EVENT IJ1 16-6-24.xlsx")
    print("Data successfully loaded!")
except FileNotFoundError as e:
    # Raising a RuntimeError completely halts Jupyter/IPython cell execution safely
    raise RuntimeError(
        "CSV files not found. Please ensure 'PERIODIC16-6-2024 LIMIT 5000.csv' "
        "and 'EVENT IJ1 16-6-24.csv' are in the exact same directory as this notebook."
    ) from e

# =========================================================
# STEP 1: PRE-PROCESS INDIVIDUAL DATASETS
# =========================================================
print("Starting Dataset Preparation...")

# Process Periodic Dataset
print("Processing Periodic Dataset...")
# Drop unwanted timestamp immediately
if "DATA_INGESTION_TIMESTAMP" in periodic_df.columns:
    periodic_df = periodic_df.drop(columns=["DATA_INGESTION_TIMESTAMP"])
    print(" - Dropped 'DATA_INGESTION_TIMESTAMP'")

# Drop columns with no values (completely null) in periodic
initial_cols_periodic = periodic_df.shape[1]
periodic_df = periodic_df.dropna(axis=1, how='all')
print(f" - Dropped {initial_cols_periodic - periodic_df.shape[1]} null columns from periodic.")

# Process Event Dataset
print("Processing Event Dataset...")
# Drop columns with no values (completely null) in event
initial_cols_event = event_df.shape[1]
# Drop Record_Type from event if present, as it was in original code
if "Record_Type" in event_df.columns:
    event_df = event_df.drop(columns=["Record_Type"])
event_df = event_df.dropna(axis=1, how='all')
print(f" - Dropped {initial_cols_event - event_df.shape[1]} null columns from event.")

# Standardize timestamps for both before decoding or looping
print("Standardizing timestamps...")
for df in [periodic_df, event_df]:
    df["PLC_TIMESTAMP"] = pd.to_datetime(
        df["PLC_TIMESTAMP"],
        utc=True
    ).dt.tz_localize(None)

# Sort event_df (needed for effective forward-filling)
event_df = event_df.sort_values("PLC_TIMESTAMP").reset_index(drop=True)


# =========================================================
# STEP 2: PLC WORD DECODING (ON PERIODIC ONLY)
# =========================================================
print("Detecting PLC Word Columns in Periodic...")

word_patterns = [
    "DRV_CMDWRD", "STTSWRD", "FLTWRD1", "FLTWRD2", "ALRMWRD",
    "SHORT_CKT_FLTWRD", "CRRNT_UNBAL_FLTWRD"
]

word_columns = [col for col in periodic_df.columns if any(pattern in col.upper() for pattern in word_patterns)]

print(f" - Found {len(word_columns)} PLC word columns.")

print("Decoding PLC words in Periodic dataset...")
expanded_periodic_df = periodic_df.copy()
bit_frames = []

for col in word_columns:
    # print(f"Decoding : {col}") # Re-enable for verbose logging
    # Convert to unsigned 16-bit integer efficiently

    values = pd.to_numeric(expanded_periodic_df[col], errors="coerce").fillna(0).astype(np.uint16).to_numpy()
    # Converts the column to numeric values.
    # Invalid values become NaN (errors="coerce").
    # Replaces NaN with 0.
    # Converts values to 16-bit unsigned integers (uint16).
    # Converts the Pandas Series to a NumPy array for faster processing.

    # Decode all 16 bits efficiently
    decoded = {
        f"{col}_BIT_{bit}": ((values >> bit) & 1).astype(np.uint8)
        for bit in range(16)
    }

    # Extracts each of the 16 bits (0–15) from every number using bitwise operations.
    # Creates a dictionary where:
    # Key = column name like WORD1_BIT_0
    # Value = array containing 0 or 1 for that bit.
    bit_frames.append(pd.DataFrame(decoded))
    # Combine all DataFrames
# bits_df = pd.concat(bit_frames, axis=1)

# Save
# bits_df.to_csv("bits.csv", index=False)

# Concatenate all at once for speed
if bit_frames:
    expanded_periodic_df = pd.concat([expanded_periodic_df] + bit_frames, axis=1)

print(f" - Decoding completed. Periodic shape: {periodic_df.shape} -> {expanded_periodic_df.shape}")
# expanded_periodic_df.to_csv("total_df.csv",index = False)

# =========================================================
# STEP 3: PREPARE EVENT DATA FOR LOOP
# =========================================================
# (Requirement: 'As-of indexing' or 'Lazy On-The-Fly' logic)
# This involves making the event data easy to look up. Pivot and sort is required.

print("Preparing Event data for chronological access...")

# Pivot Event Table to wide format (one column per tag)
event_pivot = event_df.pivot_table(
    index="PLC_TIMESTAMP",
    columns="TagName",
    values="TagValue",
    aggfunc="last"
).sort_index()

# =========================================================
# STEP 4: 60-SECOND LOOP & CLEAN DATA ALIGNMENT
# =========================================================
print("Starting 60-second Data Alignment Loop...")

start_time = expanded_periodic_df['PLC_TIMESTAMP'].min()
end_time = expanded_periodic_df['PLC_TIMESTAMP'].max()
loop_interval = timedelta(seconds=60)

current_segment_start = start_time

# Main loop that iterates through the entire dataset 60 seconds at a time
while current_segment_start <= end_time:
    current_segment_end = current_segment_start + loop_interval

    # --- a. Get Periodic Data for the 60s window ---
    periodic_60s = expanded_periodic_df[
        (expanded_periodic_df['PLC_TIMESTAMP'] >= current_segment_start) &
        (expanded_periodic_df['PLC_TIMESTAMP'] < current_segment_end)
    ].copy()

    # --- b. If this segment has periodic data, process it ---
    if not periodic_60s.empty:

        # --- FIX 1: Filter events up to the end of this window and FORWARD-FILL history ---
        # This emulates 'Lazy On-The-Fly' state persistence framework
        event_60s_pivot = event_pivot[event_pivot.index <= current_segment_end].copy()
        event_60s_pivot = event_60s_pivot.ffill() # Forward fill event states up to this point

        # --- FIX 2: Identify and drop duplicate columns (Keep Periodic version) ---
        common_cols = list(set(periodic_60s.columns).intersection(set(event_60s_pivot.columns)))
        if "PLC_TIMESTAMP" in common_cols:
            common_cols.remove("PLC_TIMESTAMP")
        event_60s_pivot = event_60s_pivot.drop(columns=common_cols)

        # Prepare for merge_asof
        periodic_60s = periodic_60s.sort_values("PLC_TIMESTAMP")
        event_aligned = event_60s_pivot.reset_index().sort_values("PLC_TIMESTAMP")

        # --- c. Merge Periodic and Event Data streams ---
        merged_60s_df = pd.merge_asof(
            periodic_60s,
            event_aligned,
            on="PLC_TIMESTAMP",
            direction="backward",
            allow_exact_matches=True
        )

        # --- FIX 3: Drop columns that are completely empty WITHIN THIS 60s SEGMENT ---
        # Prevents historical event columns that never occurred from bloating the window
        empty_cols_in_segment = merged_60s_df.columns[merged_60s_df.isna().all()]
        merged_60s_df = merged_60s_df.drop(columns=empty_cols_in_segment)

        # --- d. Store data for training variable ---
        current_60s_training_data = merged_60s_df

        # Save to validation CSV file for evaluation
        current_60s_training_data.to_csv("validate.csv", index=False)

        # -----------------------------------------------------
        # MODEL TRAINING WORKFLOW (Resets variable every loop)
        # -----------------------------------------------------
        # Model training logic using current_60s_training_data goes here...

        current_60s_training_data = None

    # Move to the next 60s window
    current_segment_start += loop_interval

print("\nData preparation and alignment loop completed.")





Data successfully loaded!
Starting Dataset Preparation...
Processing Periodic Dataset...
 - Dropped 'DATA_INGESTION_TIMESTAMP'
 - Dropped 5 null columns from periodic.
Processing Event Dataset...
 - Dropped 1 null columns from event.
Standardizing timestamps...
Detecting PLC Word Columns in Periodic...
 - Found 70 PLC word columns.
Decoding PLC words in Periodic dataset...
 - Decoding completed. Periodic shape: (5000, 224) -> (5000, 1344)
Preparing Event data for chronological access...
Starting 60-second Data Alignment Loop...

Data preparation and alignment loop completed.
